In [1]:
# Importações das bibliotecas e iniciar o Spark

import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, split, explode, regexp_extract

spark = (
    SparkSession
    .builder
    .appName("ETL-Netflix-02-Transform")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

print("Bibliotecas importadas e Spark iniciado")

Bibliotecas importadas e Spark iniciado


In [2]:
# Definindo diretórios

base_path = "../input"
bronze_path = os.path.join(base_path, "bronze")
silver_path = os.path.join(base_path, "silver")

os.makedirs(silver_path, exist_ok=True)

bronze_input_path = os.path.join(bronze_path, "netflix-bronze")

print("Diretórios definidos")

Diretórios definidos


In [3]:
# Ler os dados da camada bronze

df_bronze = spark.read.option("header", True).csv(bronze_input_path)
print("Dados carregados da camada bronze:")
df_bronze.show(5)

Dados carregados da camada bronze:
+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|       director|                cast|      country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|Kirsten Johnson|                NULL|United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|           NULL|Ama Qamata, Khosi...| South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglands|Julien Leclercq|Sami B

In [4]:
# Parte 1: Limpezas e Padronizações

for column in df_bronze.columns:
    df_bronze = df_bronze.withColumn(column, trim(col(column)))

df_silver = df_bronze.filter(
    col("country").isNotNull() & col("type").isNotNull()
)

df_silver = (
    df_silver
    .withColumn("duration_value", regexp_extract(col("duration"), r"(\d+)", 1))
    .withColumn("duration_value", col("duration_value").cast("int"))
)

print("Primeiras limpezas e padronizações conclúidas")

Primeiras limpezas e padronizações conclúidas


In [5]:
# Parte 2: Limpezas e Padronizações

df_silver = df_silver.withColumn("country", split(col("country"), ","))

df_silver = df_silver.withColumn("country", explode(col("country")))
df_silver = df_silver.withColumn("country", trim(col("country")))

for old, new in [
    ("show_id", "show_id"),
    ("type", "type"),
    ("title", "title"),
    ("director", "director"),
    ("cast", "cast"),
    ("country", "country"),
    ("date_added", "date_added"),
    ("release_year", "release_year"),
    ("rating", "rating"),
    ("duration", "duration_raw"),
    ("duration_value", "duration_value"),
    ("listed_in", "genre"),
]:
    df_silver = df_silver.withColumnRenamed(old, new)

print("Transformações concluídas:")
df_silver.show(5)
df_silver.printSchema()

Transformações concluídas:
+-------+-------+--------------------+---------------+--------------------+--------------+------------------+------------+------+------------+--------------------+--------------------+--------------+
|show_id|   type|               title|       director|                cast|       country|        date_added|release_year|rating|duration_raw|               genre|         description|duration_value|
+-------+-------+--------------------+---------------+--------------------+--------------+------------------+------------+------+------------+--------------------+--------------------+--------------+
|     s1|  Movie|Dick Johnson Is Dead|Kirsten Johnson|                NULL| United States|September 25, 2021|        2020| PG-13|      90 min|       Documentaries|As her father nea...|            90|
|     s2|TV Show|       Blood & Water|           NULL|Ama Qamata, Khosi...|  South Africa|September 24, 2021|        2021| TV-MA|   2 Seasons|International TV ...|After cros

In [6]:
# Salvar na camada silver

silver_output_path = os.path.join(silver_path, "netflix-silver")

df_silver.write.mode("overwrite").option("header", True).csv(silver_output_path)

print(f"Arquivo Silver salvo com sucesso em: {silver_output_path}")

Arquivo Silver salvo com sucesso em: ../input\silver\netflix-silver


In [7]:
# Parar o Spark

spark.stop()

print("Sessão Spark encerrada")

Sessão Spark encerrada
